In Azure, identity and access control is split across two distinct layers that work together:

- **Entra ID** — handles *authentication* (who are you?) and manages identities
- **Azure RBAC** — handles *authorisation* (what can you do?) on Azure resources
- **Azure Policy** — handles *governance* (what are you allowed to deploy?)

Understanding the boundary between these layers is essential for designing secure Azure architectures.

## Entra ID Roles vs Azure RBAC Roles

This is one of the most common points of confusion in Azure.

| | Entra ID Roles | Azure RBAC Roles |
|---|---|---|
| **Controls access to** | Entra ID features (users, groups, apps) | Azure resources (VMs, storage, databases) |
| **Examples** | Global Administrator, User Administrator | Owner, Contributor, Reader |
| **Assigned in** | Entra ID portal | Azure portal / resource scope |
| **Scope** | Tenant-wide | Management group, subscription, resource group, resource |

A **Global Administrator** in Entra ID can manage users and groups, but cannot create a VM unless also assigned an Azure RBAC role.  
A **Contributor** in Azure RBAC can create any Azure resource, but cannot manage Entra ID users.

> The two role systems are completely separate. You can elevate a Global Administrator to manage Azure subscriptions temporarily, but they are still distinct planes.

## Scope Hierarchy

Azure RBAC roles are assigned at a **scope** — a specific level in the Azure resource hierarchy. Permissions flow **downward**: a role assigned at a higher scope is inherited by all child scopes.

```
Management Group        ← broadest scope
    └── Subscription
            └── Resource Group
                    └── Resource   ← narrowest scope
```

### Example

```
Alice assigned Contributor at Subscription level
  → she can create/modify resources in ALL resource groups in that subscription

Bob assigned Reader at Resource Group "webapp-rg"
  → he can only view resources inside webapp-rg

Carol assigned Owner at a single Storage Account
  → she has full control only over that one storage account
```

Assigning roles at the **narrowest scope needed** is the principle of least privilege in Azure.

## Built-in Roles

Azure ships with **hundreds of built-in roles**. The four fundamental ones apply to every resource type:

| Role | Permissions |
|---|---|
| **Owner** | Full access to all resources + can assign RBAC roles to others |
| **Contributor** | Create and manage all resources — cannot grant access to others |
| **Reader** | View all resources — no changes |
| **User Access Administrator** | Manage RBAC role assignments only — no resource access |

### Common service-specific built-in roles

| Role | Scope |
|---|---|
| Virtual Machine Contributor | Create and manage VMs, not networking |
| Storage Blob Data Contributor | Read/write/delete blob data |
| Storage Blob Data Reader | Read blob data only |
| Key Vault Secrets Officer | Manage secrets in Key Vault |
| Key Vault Secrets User | Read secrets from Key Vault |
| AcrPull | Pull images from Azure Container Registry |
| Monitoring Reader | Read monitoring data and metrics |

> Prefer built-in roles over custom roles wherever possible — they are maintained by Microsoft and cover most scenarios.

## Role Assignments

A **role assignment** binds a security principal to a role at a specific scope. It has three parts:

```
WHO          WHAT              WHERE
──────────   ───────────────   ──────────────────────────
Alice        Contributor       /subscriptions/abc123
App (SPN)    Storage Reader    /subscriptions/abc123/resourceGroups/data-rg
Team group   VM Contributor    /subscriptions/abc123/resourceGroups/compute-rg
```

**Security principals** that can receive role assignments:
- **User** — a person in Entra ID
- **Group** — assigning to a group gives all members the role (preferred at scale)
- **Service Principal** — an application's identity
- **Managed Identity** — Azure-managed identity for a service

### Limits
- Up to **2,000 role assignments per subscription**
- Up to **500 role assignments per management group**

> **Best practice:** Assign roles to **groups** rather than individual users. When someone joins or leaves, update group membership — not role assignments.

## Custom Roles

When no built-in role fits exactly, you can create a **custom role** by specifying exactly which actions to allow or deny.

```json
{
  "Name": "VM Start/Stop Operator",
  "Description": "Can start and stop VMs but not create or delete them",
  "Actions": [
    "Microsoft.Compute/virtualMachines/start/action",
    "Microsoft.Compute/virtualMachines/deallocate/action",
    "Microsoft.Compute/virtualMachines/read"
  ],
  "NotActions": [],
  "DataActions": [],
  "NotDataActions": [],
  "AssignableScopes": [
    "/subscriptions/abc123"
  ]
}
```

| Field | Purpose |
|---|---|
| `Actions` | Management plane operations allowed (ARM operations) |
| `NotActions` | Exceptions subtracted from `Actions` |
| `DataActions` | Data plane operations allowed (reading blob data, sending queue messages) |
| `NotDataActions` | Exceptions subtracted from `DataActions` |
| `AssignableScopes` | Which subscriptions or management groups this role can be assigned in |

> Custom roles are shared across the tenant — define them once, assign them anywhere within `AssignableScopes`.

## RBAC Evaluation Logic

When an action is requested, Azure evaluates RBAC in this order:

1. **Deny assignments** — if any deny assignment covers the action, it is denied immediately, regardless of any Allow roles
2. **Allow role assignments** — if any role assignment at any scope allows the action, it is permitted
3. **Implicit deny** — if nothing allows it, it is denied by default

```
Is there a Deny assignment?  →  YES → DENIED (always wins)
                             ↓  NO
Is there an Allow assignment?  →  YES → ALLOWED
                               ↓  NO
                               → DENIED (implicit)
```

### Deny Assignments

Unlike AWS explicit denies (which you write yourself), **Azure deny assignments are created by the platform** — you cannot create them directly. They appear when:
- **Azure Blueprints** locks resources to prevent modification
- **Azure Managed Applications** protects its managed resource group

> The key difference from AWS: in Azure, multiple Allow roles accumulate — a user's effective permissions are the **union** of all their role assignments at all scopes. In AWS, permissions must be explicitly granted and deny always wins.

## Management Groups

**Management Groups** sit above subscriptions and let you organise subscriptions into a hierarchy for governance at scale.

```
Root Management Group (tenant-level)
├── MG: Platform
│   ├── Subscription: connectivity
│   └── Subscription: identity
├── MG: Landing Zones
│   ├── MG: Production
│   │   ├── Subscription: prod-app-1
│   │   └── Subscription: prod-app-2
│   └── MG: Development
│       └── Subscription: dev
└── MG: Sandbox
    └── Subscription: sandbox
```

### Why Management Groups matter

- Assign an RBAC role or Azure Policy at the Management Group level → it applies to **all subscriptions and resources** within
- Up to **6 levels** of nesting (not counting Root)
- Up to **10,000 management groups** in a single tenant
- One subscription can only have **one parent** management group

> Management Groups are Azure's equivalent of AWS Organizations OUs — the governance layer above individual subscriptions.

## Azure Policy

**Azure Policy** evaluates resources against rules and enforces organisational standards at scale. While RBAC controls *who* can take an action, Azure Policy controls *what* resources are allowed to look like.

Example policies:
- Require all resources to have a specific tag
- Deny creation of VMs larger than a certain size
- Require storage accounts to use HTTPS only
- Automatically deploy the Log Analytics agent to every new VM
- Require SQL databases to have Transparent Data Encryption enabled

Azure Policy works continuously — it evaluates **new resources** at creation time and **existing resources** on a periodic scan.

## Policy Effects

Each policy definition has an **effect** — what happens when the rule matches.

| Effect | When it runs | What it does |
|---|---|---|
| **Deny** | At creation/update time | Blocks the request outright |
| **Audit** | At creation/update + periodic scan | Allows but logs a compliance warning |
| **Append** | At creation/update time | Adds fields to the request (e.g. force-add a tag) |
| **Modify** | At creation/update time | Adds, updates, or removes properties or tags |
| **DeployIfNotExists** | After resource creation | Deploys a related resource if it is missing (e.g. diagnostic settings) |
| **AuditIfNotExists** | Periodic scan | Audits if a related resource is missing |
| **Disabled** | Never | Policy is turned off |

### Effect precedence (when multiple policies match)

```
Disabled → Append → Deny → Audit → Modify → AuditIfNotExists → DeployIfNotExists
```

> **Deny** stops the resource from being created. **Audit** lets it through but marks it non-compliant. Start with Audit when rolling out a new policy, then switch to Deny once you have addressed existing non-compliance.

## Policy Definitions

A **policy definition** is a JSON rule that describes the condition to evaluate and the effect to apply.

```json
{
  "mode": "Indexed",
  "policyRule": {
    "if": {
      "allOf": [
        { "field": "type", "equals": "Microsoft.Compute/virtualMachines" },
        { "field": "tags['Environment']", "exists": "false" }
      ]
    },
    "then": {
      "effect": "Deny"
    }
  }
}
```

This policy denies creation of any VM that is missing the `Environment` tag.

### Built-in vs custom definitions

Azure ships **hundreds of built-in policy definitions** covering security baselines, regulatory standards, and resource configuration. Browse them in the portal under Policy → Definitions.  
You can also write **custom definitions** for organisation-specific rules.

### Mode

| Mode | Evaluates |
|---|---|
| `Indexed` | Only resource types that support tags and location |
| `All` | All resource types including resource groups and subscriptions |

## Policy Initiatives (Policy Sets)

An **initiative** (also called a policy set) is a collection of related policy definitions grouped together for a common goal — like implementing a regulatory standard.

Built-in initiatives:
- **Azure Security Benchmark** — Microsoft's recommended security controls for Azure
- **CIS Microsoft Azure Foundations Benchmark** — CIS hardening guidelines
- **NIST SP 800-53** — US federal security standard
- **ISO 27001** — international information security standard
- **PCI DSS** — payment card industry standard

```
Initiative: "Require tags on resources" (custom)
├── Policy: Require 'Environment' tag
├── Policy: Require 'CostCenter' tag
└── Policy: Require 'Owner' tag
```

> Assign one initiative instead of individual policies — compliance is reported at the initiative level, giving you a single score.

## Policy Assignment & Compliance

A **policy assignment** attaches a definition or initiative to a scope (management group, subscription, or resource group), making it active.

### Assignment options

| Option | Description |
|---|---|
| **Scope** | Where the policy applies |
| **Exclusions** | Child scopes exempt from the policy |
| **Parameters** | Values passed into the policy (e.g. allowed VM SKUs list) |
| **Managed Identity** | Required for DeployIfNotExists and Modify effects — the identity that performs remediation |

### Remediation tasks

For **DeployIfNotExists** and **Modify** policies, Azure can automatically fix existing non-compliant resources through a **Remediation Task**. This uses the policy assignment's Managed Identity to make the required changes.

### Compliance dashboard

The Azure Policy compliance dashboard shows:
- Overall compliance percentage per initiative
- Non-compliant resources and policies
- Compliance history over time
- Drill-down per resource to see which policies it fails

> New resources are evaluated within **30 minutes** of creation. Existing resources are scanned on a **24-hour cycle** (or triggered manually).

## Working with RBAC and Policy via Python

In [ ]:
# pip install azure-identity azure-mgmt-authorization azure-mgmt-policyinsights

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.authorization import AuthorizationManagementClient

credential = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"

auth_client = AuthorizationManagementClient(credential, subscription_id)

# List all role assignments in the subscription
assignments = auth_client.role_assignments.list_for_subscription()
for a in assignments:
    print(f"Principal: {a.principal_id:<40} Role: {a.role_definition_id.split('/')[-1]}")

In [ ]:
# List all built-in role definitions
scope = f"/subscriptions/{subscription_id}"
roles = auth_client.role_definitions.list(scope, filter="type eq 'BuiltInRole'")

for role in roles:
    print(f"{role.role_name:<50} {role.description[:60]}")

In [ ]:
from azure.mgmt.policyinsights import PolicyInsightsClient

policy_client = PolicyInsightsClient(credential)

# Get non-compliant resources in the subscription
results = policy_client.policy_states.list_query_results_for_subscription(
    policy_states_resource="latest",
    subscription_id=subscription_id,
    query_options={"filter": "complianceState eq 'NonCompliant'"}
)

for state in results:
    print(f"{state.resource_id.split('/')[-1]:<40} Policy: {state.policy_definition_name}")

## Best Practices

| Practice | Why |
|---|---|
| Assign roles to groups, not users | Easier to manage at scale — change group membership, not role assignments |
| Use least-privilege, narrowest scope | Reduces blast radius of compromised credentials |
| Prefer built-in roles | Maintained by Microsoft; no risk of gaps in custom role definitions |
| Use Managed Identities for services | No credentials to manage or rotate |
| Start Azure Policy with Audit effect | Understand impact before enforcing Deny |
| Use initiatives over individual policies | Single compliance score; easier to assign and report |
| Apply policies at Management Group level | Consistent governance across all subscriptions |
| Use PIM for privileged RBAC roles | Just-in-time activation reduces standing privileged access |

## Summary

| Concept | Key Takeaway |
|---|---|
| Entra ID roles | Control Entra ID features — separate from Azure resource access |
| Azure RBAC | Controls access to Azure resources via role assignments at a scope |
| Scope hierarchy | Management Group → Subscription → Resource Group → Resource — permissions inherit downward |
| Built-in roles | Owner (full + assign), Contributor (full), Reader (view), User Access Admin (assign only) |
| Role assignment | WHO (principal) + WHAT (role) + WHERE (scope) — assign to groups at broadest sensible scope |
| Custom roles | JSON-defined allow/deny for Actions and DataActions |
| RBAC evaluation | Deny assignments win; otherwise union of all Allow assignments; implicit deny if nothing matches |
| Management Groups | Hierarchy above subscriptions — up to 6 levels; RBAC and Policy inherit down |
| Azure Policy | Enforces resource configuration rules — independent of who created them |
| Policy effects | Deny (blocks), Audit (logs), Modify/Append (changes), DeployIfNotExists (remediates) |
| Policy initiatives | Group of policies — assign once, compliance reported as a single score |
| Remediation tasks | Auto-fix existing non-compliant resources for DeployIfNotExists and Modify policies |